# 03 · F&B 운영변경 A2A Handoff
현장 담당 Agent의 구매요청, 신규 원부자재 생산 투입, Cost Center 변경 handoff를 Orchestration Agent가 받아, LLM이 Team delegation·A2A handoff·SAP BTP Grounding 조회를 선택하는 실습입니다.

**학습 목표:** LLM Orchestration의 delegation·A2A Message·Task·Artifact·middleware와 SAP BTP Grounding 근거를 확인합니다.  
**실행 경계:** 조직 경계는 설비팀·구매팀·제조기획·IT팀 Agent 네 Card로 공개합니다. 팀 내부 sub-agent와 모든 판단은 AI Core LLM이 입력으로부터 생성하며, 이 실습은 ERP·SAP에 실제 쓰기 작업을 하지 않습니다.

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #2e7d32; background: #e8f5e9; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #2e7d32; letter-spacing: .02em;">🖥️  A2A 서버 실행</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">A2A Protocol 1.0 JSON-RPC Task</h2>
<p style="margin: 7px 0;">먼저 별도 터미널에서 <code>python 4_agent_patterns/main.py fnb-a2a-local</code>을 실행하세요. 설비팀·구매팀·제조기획·IT팀 Agent Card와 A2A JSON-RPC Task를 준비합니다. 구매요청뿐 아니라 신규 원부자재 생산 투입과 생산라인 Cost Center 변경도 업무 시나리오로 실행할 수 있습니다. 제조기획 Agent는 생산계획·연간 OPEX·변경 Cost Center·예산 이관 신청 근거를, IT팀 Agent는 승인된 업무 변경의 ERP 데이터·변경 통제 준비 요건을 검토합니다. Team 선택과 호출 순서는 LLM이 누적 artifact를 근거로 결정합니다.</p>
  </div>
</div>


In [1]:
from pathlib import Path
import importlib
import sys
project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'shared').exists())
sys.path.insert(0, str(project_root / 'shared'))
sys.path.insert(0, str(project_root / '4_agent_patterns'))

import src.agent_to_agent.catalog as a2a_catalog_module
import src.agent_to_agent.orchestration as a2a_module
import src.agent_to_agent.ui as a2a_ui_module
a2a_catalog_module = importlib.reload(a2a_catalog_module)
a2a_module = importlib.reload(a2a_module)
a2a_ui_module = importlib.reload(a2a_ui_module)
DEFAULT_REQUEST = a2a_catalog_module.DEFAULT_REQUEST
SCENARIO_REQUESTS = a2a_catalog_module.SCENARIO_REQUESTS
list_agents = a2a_module.list_agents
run_handoff = a2a_module.run_handoff
if 'a2a_ui' in globals():
    a2a_ui.close()
a2a_ui = a2a_ui_module.launch_a2a_ui()

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅  체크포인트</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<ul style="margin: 7px 0 0; padding-left: 1.3em;"><li>네 개의 <code>/.well-known/agent-card.json</code> Card에서 조직 경계와 Skill·접속 URL을 확인한다.</li><li>구매팀, 제조기획 Agent, IT팀 Agent가 선택될 때의 TextPart·JSON DataPart와 Task ID를 확인한다.</li><li>제조기획 Agent의 생산기획·OPEX 예산·Cost Center·예산 이관 신청 검토와 IT팀 Agent의 ERP 데이터·변경관리 검토 artifact를 확인한다.</li><li>사람 승인과 외부 시스템 권한 없이는 예산 이관이나 ERP 반영을 수행하지 않는 경계를 확인한다.</li></ul>
  </div>
</div>


<hr style="border: 0; border-top: 2px solid #c7d2e1; margin: 32px 0 14px;">
<div style="margin: 0 0 18px; color: #4b5f78; font-size: .86em; font-weight: 700; letter-spacing: .04em;">🧩 아래부터 코드 베이스 실습</div>


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 심화</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
    <h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">언제 A2A를 사용할까?</h2>
    <p style="margin: 7px 0;">고정된 내부 Agent를 한 팀이 관리한다면 일반 HTTPS delegation만으로 충분합니다. URL, 요청·응답 JSON, 인증, 재시도, 작업 상태를 팀의 API 계약으로 일관되게 관리하면 A2A는 필수가 아닙니다.</p>
    <table style="width:100%; margin:10px 0 4px; border-collapse:collapse; font-size:.92em;"><thead><tr><th style="text-align:left; padding:7px; border-bottom:1px solid #c7d2e1;">필요</th><th style="text-align:left; padding:7px; border-bottom:1px solid #c7d2e1;">A2A 기능</th></tr></thead><tbody><tr><td style="padding:7px; border-bottom:1px solid #d9e2f3;">무슨 Agent인지·무엇을 하는지 발견</td><td style="padding:7px; border-bottom:1px solid #d9e2f3;"><b>Agent Card</b>의 Skill, 입출력 모드, 접속 URL</td></tr><tr><td style="padding:7px; border-bottom:1px solid #d9e2f3;">장기 작업의 상태·실패·취소 관리</td><td style="padding:7px; border-bottom:1px solid #d9e2f3;"><b>Task</b> ID와 working / completed / failed 등 표준 lifecycle</td></tr><tr><td style="padding:7px; border-bottom:1px solid #d9e2f3;">다음 Agent에 구조화 결과 전달</td><td style="padding:7px; border-bottom:1px solid #d9e2f3;"><b>Message·Artifact·DataPart</b>의 JSON·파일 전달 계약</td></tr><tr><td style="padding:7px;">서로 다른 팀·프레임워크 Agent 연결</td><td style="padding:7px;"><b>JSON-RPC/HTTP</b> 공통 계약으로 내부 구현을 숨긴 상호운용</td></tr></tbody></table>
    <p style="margin: 9px 0 0; color:#4b5f78;"><b>판단:</b> 내부의 고정된 3개 Agent라면 HTTPS handoff가 더 단순할 수 있습니다. 외부·타팀 Agent, 교체 가능한 Agent, 여러 프레임워크 연결이 필요할 때 A2A의 표준화 비용 절감 효과가 커집니다.</p>
  </div>
</div>


In [ ]:
from pathlib import Path
import sys
project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'shared').exists())
sys.path.insert(0, str(project_root / 'shared'))
sys.path.insert(0, str(project_root / '4_agent_patterns'))
from src.agent_to_agent.catalog import DEFAULT_REQUEST
from src.agent_to_agent.orchestration import list_agents, run_handoff


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);"><div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;"><h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. A2A Team Card와 내부 sub-agent 보기</h2><p style="margin: 7px 0;">A2A는 조직 경계만 공개합니다. 설비팀 안에는 현장·엔지니어·관리자 Agent, 구매팀 안에는 구매·원가·재고·MDM Agent가 있습니다. 제조기획 Agent는 생산기획·OPEX 예산·Cost Center 관리·예산 이관 신청 Agent를, IT팀 Agent는 ERP 데이터·IT 변경관리 Agent를 조율합니다.</p></div></div>


In [ ]:
for agent in list_agents():
    print(f"## {agent['agent_label']} ({agent['agent_id']})")
    print(f"설명: {agent['description']}")
    print(f"Skill: {agent['skill_id']} | URL: {agent['url']}")
    print(f"내부 sub-agent: {' → '.join(agent['sub_agents'])}")
    print(f"실행 경계: {agent['execution_boundary']}\n")


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);"><div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div><div style="margin-top: 8px; font-size: .94em; line-height: 1.6;"><h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">2. 운영변경에서 ERP 데이터 반영 준비까지</h2><p style="margin: 7px 0;">UI의 업무 시나리오는 설비 예비품 구매, 신규 원부자재의 생산계획 투입, 생산라인 Cost Center 변경을 제공합니다. LLM은 입력과 누적 artifact를 바탕으로 조달·생산계획·예산·ERP 준비 관점을 선택하며, 실제 ERP API는 호출하지 않습니다.</p></div></div>


In [ ]:
request = DEFAULT_REQUEST  # 필요하면 여기에서 실제 현장 장애 설명으로 바꾸세요.
result = run_handoff(request)
print('[A2A Context]', result['context_id'])
print('[종료 상태]', result['termination'])
print('\n[Team artifact]')
for record in result['artifacts']:
    synthesis = record['artifact']['payload'].get('team_synthesis', {})
    print(f"- {record['producer']}: {synthesis.get('summary', '분석 완료')}")
